In [1]:
#!/usr/bin/env python3
"""
PlanetScope Daily Mosaicking: Individual UDM matching + Daily mean composites
- Matches each BOA to its SPECIFIC UDM by full filename (YYYYMMDD_time_id)
- Creates improved clear mask PER IMAGE  
- Mosaics all images from same date → daily mean composite
"""

import os
import shutil
import numpy as np
import rasterio
from collections import defaultdict
from pathlib import Path

# =============================================================================
# CONFIGURATION
# =============================================================================
im_folder = Path('/mnt/CEPH_PROJECTS/Environtwin/FORCE/level2_sites_raw/MH/coregistered/')
udm_folder = Path('/mnt/CEPH_PROJECTS/Environtwin/FORCE/level2_sites_raw/MH/standard')
output_folder = Path('/mnt/CEPH_PROJECTS/Environtwin/FORCE/level2_sites_daily/03/MH/test')

output_folder.mkdir(parents=True, exist_ok=True)
nodata_val = -9999

# Log files
log_missing_pairs = output_folder / "missing_udm_pairs.log"
log_profile_mismatch = output_folder / "profile_mismatches.log"

# =============================================================================
# CORE FUNCTIONS
# =============================================================================

def extract_date(filepath):
    """Extract YYYYMMDD from filename."""
    filename = Path(filepath).stem
    if "_PLANET" not in filename:
        return None
    return filename[:8]

def append_log(log_file, msg):
    """Append to log file."""
    try:
        with open(log_file, 'a') as f:
            f.write(f"[{os.popen('date').read().strip()}] {msg}\n")
    except:
        pass

# =============================================================================
# 1. FIND INDIVIDUAL IMAGE PAIRS BY FULL FILENAME
# =============================================================================

def extract_prefix(filepath):
    """Extract common prefix: YYYYMMDD_HHMMSS_xx_xxxx from PlanetScope filenames."""
    filename = os.path.basename(filepath)
    stem = os.path.splitext(filename)[0]  # Remove .tif/.bsq
    if "_PLANET" not in stem:
        return None
    return stem.split("_PLANET")[0]  # "20211130_091749_81_2423_PLANET_*" → "20211130_091749_81_2423"

def find_individual_pairs():
    """Match BOA-UDM using extract_prefix()!"""
    print("Matching BOA-UDM pairs by common prefix...")
    
    # Create {prefix: filepath} dictionaries
    boa_dict = {}
    udm_dict = {}
    
    # Scan BOA files
    for f in im_folder.rglob("*_PLANET_BOA.bsq"):
        prefix = extract_prefix(f)
        if prefix:
            boa_dict[prefix] = f
            print(f"BOA: {f.name} → {prefix}")
    
    # Scan UDM files  
    for f in udm_folder.rglob("*_PLANET_udm2_buffer.tif"):
        prefix = extract_prefix(f)
        if prefix:
            udm_dict[prefix] = f
            print(f"UDM: {f.name} → {prefix}")
    
    # Perfect 1:1 matching!
    pairs = []
    date_groups = defaultdict(list)
    
    for prefix, boa_fp in boa_dict.items():
        if prefix in udm_dict:
            udm_fp = udm_dict[prefix]
            pairs.append((boa_fp, udm_fp))
            
            # Extract date for grouping
            date = prefix[:8]  # YYYYMMDD
            date_groups[date].append((boa_fp, udm_fp))
            print(f"✓ MATCH: {prefix}")
        else:
            print(f"NO UDM for prefix: {prefix}")
    
    print(f"\n {len(pairs)} PERFECT PAIRS across {len(date_groups)} dates")
    return dict(date_groups)


# =============================================================================
# 2. PROCESS ALL IMAGES FOR ONE DATE → DAILY MOSAIC
# =============================================================================

def process_daily_mosaic(date, image_pairs):
    """Process all image pairs for one date → create daily mean mosaic."""
    n_images = len(image_pairs)
    print(f"\n{'='*80}")
    print(f"DATE {date}: MOSAICKING {n_images} IMAGE PAIRS")
    print(f"{'='*80}")
    
    # Output files
    out_boa = output_folder / f"{date}_PLANET_BOA.tif"
    out_udm = output_folder / f"{date}_PLANET_udm2_mask.tif"
    out_cnt = output_folder / f"{date}_PLANET_count.tif"
    
    # Skip if exists
    if all(f.exists() for f in [out_boa, out_udm, out_cnt]):
        print(f"⏭️  Skipping {date} (already processed)")
        return
    
    # Get reference profile from first image
    ref_boa, ref_udm = image_pairs[0]
     
    # Setup output profiles
    with rasterio.open(ref_boa) as ref:
        height, width = ref.height, ref.width
        bands = ref.count
        ref_crs = ref.crs
        ref_transform = ref.transform
        band_descriptions = ref.descriptions
        boa_profile = ref.profile.copy()
        boa_profile.update(dtype="float32", nodata=nodata_val, compress="deflate")

    with rasterio.open(ref_udm) as refm:
        udm_bands = refm.count   
        udm_profile = refm.profile.copy()
        udm_profile.update(
            count=udm_bands,
            dtype="int16",
            compress="deflate",
            nodata=-9999
        )

    cnt_profile = boa_profile.copy()
    cnt_profile.update(count=1, dtype="uint16", nodata=0)
    
    print(f" Scene: {width}x{height}, BOA={bands}b, UDM={udm_bands}b")
    
    # =============================================================================
    # PROCESS EACH IMAGE PAIR INDIVIDUALLY
    # =============================================================================
    
    if n_images == 1:
        print("Single image → applying mask and writing directly")

        boa_fp, udm_fp = image_pairs[0]

        with rasterio.open(boa_fp) as boa, rasterio.open(udm_fp) as udm:

            masked_full = np.full((bands, height, width),
                                nodata_val,
                                dtype=np.int16)

            udm_full = np.full((udm_bands, height, width),
                            -9999,
                            dtype=np.int16)
            
            count_full = np.zeros((height, width), dtype=np.uint16)

            for (r0, c0), win in boa.block_windows(1):
                rr = slice(r0, r0 + win.height)
                cc = slice(c0, c0 + win.width)

                boa_win = boa.read(window=win)
                udm_win = udm.read(window=win)
          
                # CREATE IMPROVED CLEAR MASK FOR THIS SPECIFIC IMAGE
                clear = udm_win[10, :, :] 

                if boa.nodata is not None:
                    boa_valid = np.all(boa_win != boa.nodata, axis=0)  # shape (rows, cols)

                final_mask = (clear == 1) & boa_valid

                masked_win = np.where(final_mask[None, :, :], boa_win, nodata_val)  # broadcast over bands
                masked_full[:, rr, cc] = masked_win

                # --- COUNT ---
                count_full[rr, cc] = final_mask.astype(np.int16)

        
        # Write BOA
        with rasterio.open(out_boa, "w", **boa_profile) as dst:
            dst.write(masked_full)
            # Restore band descriptions from original image
            for i, desc in enumerate(band_descriptions, start=1):
                if desc:
                    dst.set_band_description(i, desc)

        # Write UDM (10 original + new clear)
        with rasterio.open(out_udm, "w", **udm_profile) as dst:
            dst.write(udm_win)
            band_names = [
                    'clear', 'snow', 'shadow', 'light_haze', 'heavy_haze',
                    'cloud', 'confidence', 'udm2_unusable',
                    'cloud_buffer', 'shadow_buffer', 'new_clear'
                ]
            for idx, name in enumerate(band_names, start=1):
                    dst.set_band_description(idx, name)
    
        # write count
        with rasterio.open(out_cnt, "w", **cnt_profile) as dst:
            dst.write(count_full, 1)
            dst.set_band_description(1, "count")      
        

        print(f"{date}: single masked BOA + extended UDM written")
        return                
    
    # =============================================================================
    # CREATE FINAL DAILY MOSAIC
    # =============================================================================
    
    print(f"{n_images} images → computing masked mean mosaic")

    # Initialize accumulators
    sum_boa  = np.zeros((bands, height, width), dtype=np.float64)
    cnt_boa  = np.zeros((bands, height, width), dtype=np.int16)
    cnt_img  = np.zeros((height, width), dtype=np.int16)
    sum_udm  = np.zeros((udm_bands, height, width), dtype=np.int16)

    # ------------------------
    # Loop over all image pairs
    # ------------------------
    for (boa_fp, udm_fp) in image_pairs:
        print(f"Processing {boa_fp.name}")

        with rasterio.open(boa_fp) as boa, rasterio.open(udm_fp) as udm:

            for (r0, c0), win in boa.block_windows(1):

                rr = slice(r0, r0 + win.height)
                cc = slice(c0, c0 + win.width)

                boa_win = boa.read(window=win)
                udm_win = udm.read(window=win)

                # ---------------------------------------
                # Use improved clear band (band 11 = index 10)
                # ---------------------------------------
                clear = udm_win[10].astype(np.int16)

                # ---------------------------------------
                # Accumulate FULL UDM stack
                # ---------------------------------------
                sum_udm[:, rr, cc] += udm_win.astype(np.int16)

                # ---------------------------------------
                # BOA validity check
                # ---------------------------------------
                boa_valid = np.all(
                    boa_win != (boa.nodata if boa.nodata is not None else nodata_val),
                    axis=0
                )

                final_mask = clear & boa_valid

                # Accumulate counts
                cnt_img[rr, cc] += final_mask
                cnt_boa[:, rr, cc] += final_mask

                # --- Masked BOA ---
                masked_win = np.where(
                    final_mask[None, :, :],
                    boa_win,
                    0  # use 0 for accumulation, nodata applied later
                )

                # --- Accumulate BOA sum and per-band counts ---
                sum_boa[:, rr, cc] += masked_win

    # ------------------------
    # Compute daily BOA mean
    # ------------------------
    with np.errstate(divide='ignore', invalid='ignore'):
        daily_mean = np.where(
            cnt_boa > 0,
            sum_boa / cnt_boa,
            nodata_val
        ).astype(np.int16)

    # ------------------------
    # Clip UDM to 0/1
    # ------------------------
    udm_final = np.where(sum_udm > 1, 1, sum_udm).astype(np.int16)

    # ------------------------
    # Write BOA
    # ------------------------
    with rasterio.open(out_boa, "w", **boa_profile) as dst:
        dst.write(daily_mean)
        # Restore band descriptions from original image
        for i, desc in enumerate(band_descriptions, start=1):
            if desc:
                dst.set_band_description(i, desc)

    # ------------------------
    # Write UDM (original bands + improved clear)
    # ------------------------
    with rasterio.open(out_udm, "w", **udm_profile) as dst:
        dst.write(udm_final)
        band_names = [
                    'clear', 'snow', 'shadow', 'light_haze', 'heavy_haze',
                    'cloud', 'confidence', 'udm2_unusable',
                    'cloud_buffer', 'shadow_buffer', 'new_clear'
                ]
        for idx, name in enumerate(band_names, start=1):
                    dst.set_band_description(idx, name)

    # ------------------------
    # Write count image
    # ------------------------
    with rasterio.open(out_cnt, "w", **cnt_profile) as dst:
        dst.write(cnt_img, 1)
        dst.set_band_description(1, "count")

    print(f" {date}: BOA mosaic + UDM + count written")


In [2]:
# Test ONLY 20180812
date_groups = find_individual_pairs()
if '20180406' in date_groups:
    process_daily_mosaic('20180406', date_groups['20180406'])
else:
    print("❌ No 20170611 data found!")

Matching BOA-UDM pairs by common prefix...
BOA: 20200728_095136_1012_PLANET_BOA.bsq → 20200728_095136_1012
BOA: 20250628_104022_39_254a_PLANET_BOA.bsq → 20250628_104022_39_254a
BOA: 20221007_091020_01_2447_PLANET_BOA.bsq → 20221007_091020_01_2447
BOA: 20170423_093046_0e2f_PLANET_BOA.bsq → 20170423_093046_0e2f
BOA: 20190328_094908_1024_PLANET_BOA.bsq → 20190328_094908_1024
BOA: 20190620_094917_0f34_PLANET_BOA.bsq → 20190620_094917_0f34
BOA: 20210823_094617_101f_PLANET_BOA.bsq → 20210823_094617_101f
BOA: 20241124_102651_30_24ae_PLANET_BOA.bsq → 20241124_102651_30_24ae
BOA: 20201114_092604_24_222b_PLANET_BOA.bsq → 20201114_092604_24_222b
BOA: 20250618_104001_23_254a_PLANET_BOA.bsq → 20250618_104001_23_254a
BOA: 20200423_095444_101f_PLANET_BOA.bsq → 20200423_095444_101f
BOA: 20240324_093258_27_242d_PLANET_BOA.bsq → 20240324_093258_27_242d
BOA: 20171109_093346_1022_PLANET_BOA.bsq → 20171109_093346_1022
BOA: 20210809_095058_1039_PLANET_BOA.bsq → 20210809_095058_1039
BOA: 20180602_094009_0f1b